## Setup and Imports

In [ ]:
# Standard library imports
import sys
import warnings
from pathlib import Path

# Visualization
import matplotlib.pyplot as plt

# MLflow
import mlflow
import mlflow.pytorch
import numpy as np

# Data manipulation
import pandas as pd
import seaborn as sns

# Machine Learning
import torch

# Configuration
warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-darkgrid")
sns.set_palette("husl")

# Set display options
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:.4f}".format)

# Add project modules to path
project_root = Path().absolute().parent
sys.path.insert(0, str(project_root))

# Import custom modules
from modules.models.autoencoder import AutoEncoder
from modules.utils.dataloader import DataframeDataset

print("Libraries imported successfully")
print(f"Project root: {project_root}")
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {torch.device('cuda' if torch.cuda.is_available() else 'cpu')}")

## 1. Model Loading and Setup

In [ ]:
# Configuration
MODEL_DIR = project_root / "data" / "mlflow"  # Models stored in MLflow artifacts
DATA_DIR = project_root / "data" / "output"  # Preprocessed data location
MLFLOW_TRACKING_URI = "http://localhost:5001"

# Set MLflow tracking URI
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
print(f"MLflow tracking URI: {MLFLOW_TRACKING_URI}")

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# List available models
if MODEL_DIR.exists():
    # Models are registered in MLflow, access via MLflow API
    # For local artifacts, check MLflow artifacts directory structure
    model_files = list(MODEL_DIR.glob("**/*.pth"))  # Search recursively
    print(f"Found {len(model_files)} trained models:")
    for model_file in model_files[:10]:  # Show first 10
        print(f"  - {model_file.relative_to(MODEL_DIR)}")
    if len(model_files) > 10:
        print(f"  ... and {len(model_files) - 10} more")
else:
    print(f"Model directory not found: {MODEL_DIR}")
    print("\nPlease train models first:")
    print("python dfp-demo/pipelines/run_training_cli.py --config dfp-demo/config/pipeline.yaml")
    model_files = []

In [ ]:
# Load a sample model to understand structure
if model_files:
    sample_model_path = model_files[0]
    print(f"Loading sample model: {sample_model_path.name}")

    # Load model state
    checkpoint = torch.load(sample_model_path, map_location=device)

    print("\nModel checkpoint keys:")
    for key in checkpoint.keys():
        print(f"  - {key}")

    # Display model configuration
    if "config" in checkpoint:
        print("\nModel configuration:")
        for key, value in checkpoint["config"].items():
            print(f"  {key}: {value}")

## 2. Reconstruction Error Computation

In [ ]:
def compute_reconstruction_error(model, data_loader, device):
    """
    Compute reconstruction error for all samples.

    Args:
        model: Trained AutoEncoder model
        data_loader: DataLoader with test data
        device: torch device (cpu/cuda)

    Returns:
        reconstruction_errors: Array of MSE for each sample
        feature_errors: Array of per-feature errors
    """
    model.eval()
    reconstruction_errors = []
    feature_errors = []

    with torch.no_grad():
        for batch in data_loader:
            inputs = batch.to(device)
            outputs = model(inputs)

            # Compute MSE per sample
            mse_per_sample = torch.mean((inputs - outputs) ** 2, dim=1)
            reconstruction_errors.extend(mse_per_sample.cpu().numpy())

            # Compute per-feature errors
            feature_error = (inputs - outputs) ** 2
            feature_errors.append(feature_error.cpu().numpy())

    reconstruction_errors = np.array(reconstruction_errors)
    feature_errors = np.vstack(feature_errors)

    return reconstruction_errors, feature_errors


print("Reconstruction error computation function defined")

In [ ]:
# Load test data for a specific user
if DATA_DIR.exists() and model_files:
    test_files = list(DATA_DIR.glob("*_test.parquet"))

    if test_files:
        print(f"Found {len(test_files)} test files")

        # Select a test file
        test_file = test_files[0]
        print(f"\nLoading test data: {test_file.name}")

        test_df = pd.read_parquet(test_file)
        print(f"Test samples: {len(test_df)}")
        print(f"Features: {len(test_df.columns)}")

        # Create dataset
        test_dataset = DataframeDataset(test_df)
        test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=64, shuffle=False)

        print(f"Test batches: {len(test_loader)}")
    else:
        print("No test files found")
        test_df = None
else:
    print("Data directory not found or no models available")
    test_df = None

In [ ]:
# Compute reconstruction errors
if test_df is not None and model_files:
    # Find matching model
    username = test_file.stem.replace("_test", "")
    model_path = MODEL_DIR / f"{username}.pth"

    if model_path.exists():
        print(f"Loading model for user: {username}")

        # Load model
        checkpoint = torch.load(model_path, map_location=device)

        # Initialize model
        input_size = len(test_df.columns)
        config = checkpoint.get("config", {})

        model = AutoEncoder(
            input_size=input_size,
            encoder_layers=config.get("encoder_layers", [128, 64]),
            latent_dim=config.get("latent_dim", 32),
            activation=config.get("activation", "relu"),
            dropout_rate=config.get("dropout_rate", 0.2),
        ).to(device)

        model.load_state_dict(checkpoint["model_state_dict"])
        print("Model loaded successfully")

        # Compute errors
        print("\nComputing reconstruction errors...")
        reconstruction_errors, feature_errors = compute_reconstruction_error(model, test_loader, device)

        print(f"Reconstruction errors computed: {len(reconstruction_errors)} samples")
        print(f"Feature errors shape: {feature_errors.shape}")
    else:
        print(f"Model not found: {model_path}")
        reconstruction_errors = None

## 3. Error Distribution Analysis

In [ ]:
# Statistical summary
if reconstruction_errors is not None:
    print("Reconstruction Error Statistics:")
    print(f"  Mean: {np.mean(reconstruction_errors):.6f}")
    print(f"  Median: {np.median(reconstruction_errors):.6f}")
    print(f"  Std Dev: {np.std(reconstruction_errors):.6f}")
    print(f"  Min: {np.min(reconstruction_errors):.6f}")
    print(f"  Max: {np.max(reconstruction_errors):.6f}")
    print("  \nPercentiles:")
    for p in [90, 95, 99, 99.5, 99.9]:
        val = np.percentile(reconstruction_errors, p)
        print(f"    {p}th: {val:.6f}")

In [ ]:
# Histogram of reconstruction errors
if reconstruction_errors is not None:
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    # Full distribution
    axes[0].hist(reconstruction_errors, bins=50, edgecolor="black", alpha=0.7)
    axes[0].axvline(
        np.mean(reconstruction_errors),
        color="red",
        linestyle="--",
        linewidth=2,
        label=f"Mean: {np.mean(reconstruction_errors):.6f}",
    )
    axes[0].axvline(
        np.median(reconstruction_errors),
        color="green",
        linestyle="--",
        linewidth=2,
        label=f"Median: {np.median(reconstruction_errors):.6f}",
    )
    axes[0].set_xlabel("Reconstruction Error (MSE)", fontsize=12)
    axes[0].set_ylabel("Frequency", fontsize=12)
    axes[0].set_title("Reconstruction Error Distribution", fontsize=14, fontweight="bold")
    axes[0].legend(fontsize=10)
    axes[0].grid(True, alpha=0.3)

    # Log scale
    axes[1].hist(np.log10(reconstruction_errors + 1e-10), bins=50, edgecolor="black", alpha=0.7)
    axes[1].set_xlabel("Log10(Reconstruction Error)", fontsize=12)
    axes[1].set_ylabel("Frequency", fontsize=12)
    axes[1].set_title("Reconstruction Error Distribution (Log Scale)", fontsize=14, fontweight="bold")
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

In [ ]:
# Box plot and violin plot
if reconstruction_errors is not None:
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    # Box plot
    axes[0].boxplot(reconstruction_errors, vert=True)
    axes[0].set_ylabel("Reconstruction Error (MSE)", fontsize=12)
    axes[0].set_title("Reconstruction Error Box Plot", fontsize=14, fontweight="bold")
    axes[0].grid(True, alpha=0.3, axis="y")

    # Violin plot
    parts = axes[1].violinplot([reconstruction_errors], vert=True, showmeans=True, showmedians=True)
    axes[1].set_ylabel("Reconstruction Error (MSE)", fontsize=12)
    axes[1].set_title("Reconstruction Error Violin Plot", fontsize=14, fontweight="bold")
    axes[1].grid(True, alpha=0.3, axis="y")

    plt.tight_layout()
    plt.show()

## 4. Normal vs Anomaly Separation

In [ ]:
# Define threshold using multiple methods
if reconstruction_errors is not None:
    # Method 1: Mean + k*StdDev
    mean_error = np.mean(reconstruction_errors)
    std_error = np.std(reconstruction_errors)

    thresholds = {
        "Mean + 2 Std": mean_error + 2 * std_error,
        "Mean + 3 Std": mean_error + 3 * std_error,
        "95th Percentile": np.percentile(reconstruction_errors, 95),
        "99th Percentile": np.percentile(reconstruction_errors, 99),
        "99.5th Percentile": np.percentile(reconstruction_errors, 99.5),
    }

    print("Anomaly Detection Thresholds:")
    for method, threshold in thresholds.items():
        n_anomalies = np.sum(reconstruction_errors > threshold)
        pct_anomalies = (n_anomalies / len(reconstruction_errors)) * 100
        print(f"  {method}: {threshold:.6f} ({n_anomalies} anomalies, {pct_anomalies:.2f}%)")

In [ ]:
# Visualize thresholds
if reconstruction_errors is not None:
    plt.figure(figsize=(15, 6))

    plt.hist(reconstruction_errors, bins=100, edgecolor="black", alpha=0.6, label="Reconstruction Errors")

    colors = ["red", "orange", "green", "blue", "purple"]
    for idx, (method, threshold) in enumerate(thresholds.items()):
        plt.axvline(threshold, color=colors[idx], linestyle="--", linewidth=2, label=method)

    plt.xlabel("Reconstruction Error (MSE)", fontsize=12)
    plt.ylabel("Frequency", fontsize=12)
    plt.title("Reconstruction Error Distribution with Threshold Candidates", fontsize=14, fontweight="bold")
    plt.legend(fontsize=10, loc="upper right")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
# Identify anomalies using 99th percentile threshold
if reconstruction_errors is not None:
    threshold = thresholds["99th Percentile"]
    is_anomaly = reconstruction_errors > threshold

    print(f"Using threshold: {threshold:.6f} (99th percentile)")
    print("\nClassification Results:")
    print(f"  Normal samples: {np.sum(~is_anomaly)} ({(np.sum(~is_anomaly) / len(is_anomaly)) * 100:.2f}%)")
    print(f"  Anomaly samples: {np.sum(is_anomaly)} ({(np.sum(is_anomaly) / len(is_anomaly)) * 100:.2f}%)")

    # Statistics by class
    print("\nNormal samples statistics:")
    normal_errors = reconstruction_errors[~is_anomaly]
    print(f"  Mean error: {np.mean(normal_errors):.6f}")
    print(f"  Max error: {np.max(normal_errors):.6f}")

    print("\nAnomaly samples statistics:")
    anomaly_errors = reconstruction_errors[is_anomaly]
    print(f"  Mean error: {np.mean(anomaly_errors):.6f}")
    print(f"  Min error: {np.min(anomaly_errors):.6f}")

## 5. Threshold Sensitivity Analysis

In [ ]:
# Analyze impact of different thresholds
if reconstruction_errors is not None:
    percentiles = np.arange(90, 100, 0.5)
    threshold_values = np.percentile(reconstruction_errors, percentiles)
    anomaly_rates = [(reconstruction_errors > t).mean() * 100 for t in threshold_values]

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    # Threshold vs Anomaly Rate
    axes[0].plot(percentiles, anomaly_rates, linewidth=2, marker="o", markersize=4)
    axes[0].set_xlabel("Percentile Threshold", fontsize=12)
    axes[0].set_ylabel("Anomaly Rate (%)", fontsize=12)
    axes[0].set_title("Threshold Sensitivity: Percentile vs Anomaly Rate", fontsize=14, fontweight="bold")
    axes[0].grid(True, alpha=0.3)
    axes[0].axhline(1, color="red", linestyle="--", linewidth=1, label="1% anomaly rate")
    axes[0].axhline(5, color="orange", linestyle="--", linewidth=1, label="5% anomaly rate")
    axes[0].legend(fontsize=10)

    # Threshold value vs Anomaly Rate
    axes[1].plot(threshold_values, anomaly_rates, linewidth=2, marker="o", markersize=4)
    axes[1].set_xlabel("Threshold Value (MSE)", fontsize=12)
    axes[1].set_ylabel("Anomaly Rate (%)", fontsize=12)
    axes[1].set_title("Threshold Sensitivity: Value vs Anomaly Rate", fontsize=14, fontweight="bold")
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

In [ ]:
# Recommendation for threshold selection
if reconstruction_errors is not None:
    print("Threshold Selection Recommendations:")
    print("\n1. For HIGH PRECISION (few false positives):")
    high_precision_threshold = np.percentile(reconstruction_errors, 99.5)
    hp_anomaly_rate = (reconstruction_errors > high_precision_threshold).mean() * 100
    print(f"   Use 99.5th percentile: {high_precision_threshold:.6f}")
    print(f"   Expected anomaly rate: {hp_anomaly_rate:.2f}%")

    print("\n2. For BALANCED approach:")
    balanced_threshold = np.percentile(reconstruction_errors, 99)
    bal_anomaly_rate = (reconstruction_errors > balanced_threshold).mean() * 100
    print(f"   Use 99th percentile: {balanced_threshold:.6f}")
    print(f"   Expected anomaly rate: {bal_anomaly_rate:.2f}%")

    print("\n3. For HIGH RECALL (catch more anomalies):")
    high_recall_threshold = np.percentile(reconstruction_errors, 95)
    hr_anomaly_rate = (reconstruction_errors > high_recall_threshold).mean() * 100
    print(f"   Use 95th percentile: {high_recall_threshold:.6f}")
    print(f"   Expected anomaly rate: {hr_anomaly_rate:.2f}%")

## 6. Per-User Performance Metrics

In [ ]:
# Compute metrics across multiple users
if test_files and model_files:
    user_metrics = []

    # Process up to 10 users
    for test_file in test_files[:10]:
        username = test_file.stem.replace("_test", "")
        model_path = MODEL_DIR / f"{username}.pth"

        if not model_path.exists():
            continue

        try:
            # Load data
            test_df_user = pd.read_parquet(test_file)
            test_dataset_user = DataframeDataset(test_df_user)
            test_loader_user = torch.utils.data.DataLoader(test_dataset_user, batch_size=64, shuffle=False)

            # Load model
            checkpoint = torch.load(model_path, map_location=device)
            input_size = len(test_df_user.columns)
            config = checkpoint.get("config", {})

            model_user = AutoEncoder(
                input_size=input_size,
                encoder_layers=config.get("encoder_layers", [128, 64]),
                latent_dim=config.get("latent_dim", 32),
                activation=config.get("activation", "relu"),
                dropout_rate=config.get("dropout_rate", 0.2),
            ).to(device)

            model_user.load_state_dict(checkpoint["model_state_dict"])

            # Compute errors
            errors_user, _ = compute_reconstruction_error(model_user, test_loader_user, device)

            # Store metrics
            user_metrics.append(
                {
                    "username": username,
                    "n_samples": len(errors_user),
                    "mean_error": np.mean(errors_user),
                    "median_error": np.median(errors_user),
                    "std_error": np.std(errors_user),
                    "min_error": np.min(errors_user),
                    "max_error": np.max(errors_user),
                    "p95": np.percentile(errors_user, 95),
                    "p99": np.percentile(errors_user, 99),
                }
            )

        except Exception as e:
            print(f"Error processing {username}: {e}")
            continue

    if user_metrics:
        user_metrics_df = pd.DataFrame(user_metrics)
        print(f"\nProcessed {len(user_metrics_df)} users")
        print("\nPer-User Reconstruction Error Metrics:")
        print(user_metrics_df.to_string(index=False))
    else:
        user_metrics_df = None
else:
    user_metrics_df = None

In [ ]:
# Visualize per-user metrics
if user_metrics_df is not None and len(user_metrics_df) > 0:
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))

    # Mean error
    axes[0, 0].barh(range(len(user_metrics_df)), user_metrics_df["mean_error"], edgecolor="black", alpha=0.7)
    axes[0, 0].set_yticks(range(len(user_metrics_df)))
    axes[0, 0].set_yticklabels(user_metrics_df["username"])
    axes[0, 0].set_xlabel("Mean Reconstruction Error", fontsize=12)
    axes[0, 0].set_title("Mean Reconstruction Error by User", fontsize=14, fontweight="bold")
    axes[0, 0].grid(True, alpha=0.3, axis="x")

    # 99th percentile
    axes[0, 1].barh(range(len(user_metrics_df)), user_metrics_df["p99"], edgecolor="black", alpha=0.7, color="orange")
    axes[0, 1].set_yticks(range(len(user_metrics_df)))
    axes[0, 1].set_yticklabels(user_metrics_df["username"])
    axes[0, 1].set_xlabel("99th Percentile Error", fontsize=12)
    axes[0, 1].set_title("99th Percentile Reconstruction Error by User", fontsize=14, fontweight="bold")
    axes[0, 1].grid(True, alpha=0.3, axis="x")

    # Standard deviation
    axes[1, 0].barh(
        range(len(user_metrics_df)), user_metrics_df["std_error"], edgecolor="black", alpha=0.7, color="green"
    )
    axes[1, 0].set_yticks(range(len(user_metrics_df)))
    axes[1, 0].set_yticklabels(user_metrics_df["username"])
    axes[1, 0].set_xlabel("Std Dev of Error", fontsize=12)
    axes[1, 0].set_title("Error Variability by User", fontsize=14, fontweight="bold")
    axes[1, 0].grid(True, alpha=0.3, axis="x")

    # Sample size
    axes[1, 1].barh(
        range(len(user_metrics_df)), user_metrics_df["n_samples"], edgecolor="black", alpha=0.7, color="purple"
    )
    axes[1, 1].set_yticks(range(len(user_metrics_df)))
    axes[1, 1].set_yticklabels(user_metrics_df["username"])
    axes[1, 1].set_xlabel("Number of Test Samples", fontsize=12)
    axes[1, 1].set_title("Test Sample Count by User", fontsize=14, fontweight="bold")
    axes[1, 1].grid(True, alpha=0.3, axis="x")

    plt.tight_layout()
    plt.show()

## 7. Feature-Level Error Analysis

In [ ]:
# Analyze per-feature errors
if feature_errors is not None and test_df is not None:
    feature_names = test_df.columns.tolist()
    mean_feature_errors = np.mean(feature_errors, axis=0)

    feature_error_df = pd.DataFrame({"Feature": feature_names, "Mean Error": mean_feature_errors}).sort_values(
        "Mean Error", ascending=False
    )

    print("Top 20 Features by Reconstruction Error:")
    print(feature_error_df.head(20).to_string(index=False))

In [ ]:
# Visualize feature-level errors
if feature_errors is not None and test_df is not None:
    top_features = feature_error_df.head(20)

    plt.figure(figsize=(15, 8))
    plt.barh(range(len(top_features)), top_features["Mean Error"], edgecolor="black", alpha=0.7)
    plt.yticks(range(len(top_features)), top_features["Feature"])
    plt.xlabel("Mean Reconstruction Error", fontsize=12)
    plt.ylabel("Feature", fontsize=12)
    plt.title("Top 20 Features by Reconstruction Error", fontsize=14, fontweight="bold")
    plt.grid(True, alpha=0.3, axis="x")
    plt.tight_layout()
    plt.show()

In [ ]:
# Feature error distribution heatmap (for top features)
if feature_errors is not None and test_df is not None:
    top_feature_indices = feature_error_df.head(15).index
    top_feature_errors = feature_errors[:, top_feature_indices]
    top_feature_names = feature_error_df.head(15)["Feature"].tolist()

    # Sample 100 random rows for visualization
    sample_indices = np.random.choice(len(top_feature_errors), min(100, len(top_feature_errors)), replace=False)
    sample_errors = top_feature_errors[sample_indices]

    plt.figure(figsize=(14, 10))
    sns.heatmap(
        sample_errors.T,
        cmap="YlOrRd",
        cbar_kws={"label": "Squared Error"},
        yticklabels=top_feature_names,
        xticklabels=False,
    )
    plt.xlabel("Sample Index", fontsize=12)
    plt.ylabel("Feature", fontsize=12)
    plt.title(
        "Feature-Level Reconstruction Error Heatmap (Top 15 Features, 100 Samples)", fontsize=14, fontweight="bold"
    )
    plt.tight_layout()
    plt.show()

## Summary and Recommendations

In [ ]:
print("=" * 80)
print("RECONSTRUCTION ERROR ANALYSIS SUMMARY")
print("=" * 80)

if reconstruction_errors is not None:
    print("\n1. ERROR DISTRIBUTION")
    print(f"   Mean reconstruction error: {np.mean(reconstruction_errors):.6f}")
    print(f"   Median reconstruction error: {np.median(reconstruction_errors):.6f}")
    print(f"   Standard deviation: {np.std(reconstruction_errors):.6f}")
    print(f"   99th percentile: {np.percentile(reconstruction_errors, 99):.6f}")

    print("\n2. RECOMMENDED THRESHOLDS")
    print(f"   Conservative (99.5%ile): {np.percentile(reconstruction_errors, 99.5):.6f}")
    print(f"   Balanced (99%ile): {np.percentile(reconstruction_errors, 99):.6f}")
    print(f"   Aggressive (95%ile): {np.percentile(reconstruction_errors, 95):.6f}")

if user_metrics_df is not None and len(user_metrics_df) > 0:
    print("\n3. PER-USER PERFORMANCE")
    print(f"   Users analyzed: {len(user_metrics_df)}")
    print(f"   Average mean error: {user_metrics_df['mean_error'].mean():.6f}")
    print(f"   Error variability (avg std): {user_metrics_df['std_error'].mean():.6f}")
    print(f"   Best performing user: {user_metrics_df.loc[user_metrics_df['mean_error'].idxmin(), 'username']}")
    print(f"   User needing attention: {user_metrics_df.loc[user_metrics_df['mean_error'].idxmax(), 'username']}")

if feature_errors is not None and test_df is not None:
    print("\n4. FEATURE-LEVEL INSIGHTS")
    print(f"   Most difficult feature: {feature_error_df.iloc[0]['Feature']}")
    print(f"   Error: {feature_error_df.iloc[0]['Mean Error']:.6f}")
    print(f"   Features with high error (>0.01): {(feature_error_df['Mean Error'] > 0.01).sum()}")

print("\n5. RECOMMENDATIONS")
print("   - Use 99th percentile as default anomaly threshold")
print("   - Monitor users with consistently high reconstruction errors")
print("   - Investigate features with disproportionately high errors")
print("   - Consider per-user threshold adaptation for production")
print("   - Validate threshold selection with labeled anomaly data")

print("\n" + "=" * 80)